# Chapter 7. 심층강화학습 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter07_dqn_tricks.ipynb)

책 본문: [Chapter 7](https://smhanlab.com/book-ml/kor/ml2/chapter07.html)

DQN 자체(신경망 학습)를 처음부터 돌리는 대신, DQN을 실제로 안정적으로
만드는 두 공학적 장치 — **경험 재현(Experience Replay)**과 **타겟
네트워크(Target Network)** — 를 작은 예제로 직접 확인합니다.

## 1. 경험 재현 버퍼 (책 7.4절, 연습문제 예제 그대로)

In [ ]:
import random

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = []
        self.capacity = capacity

    def push(self, transition):
        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)
        self.buffer.append(transition)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

buf = ReplayBuffer(capacity=3)
buf.push(("s1", "a1", 1, "s2"))
buf.push(("s2", "a2", 0, "s3"))
buf.push(("s3", "a3", 1, "s4"))
buf.push(("s4", "a4", 0, "s5"))  # 버퍼가 가득 차서 첫 번째 항목이 밀려남

print(len(buf.buffer))  # 3
print(buf.buffer[0])    # ('s2', 'a2', 0, 's3')
assert len(buf.buffer) == 3
assert buf.buffer[0] == ("s2", "a2", 0, "s3")
print("FIFO로 오래된 경험이 정상적으로 밀려남을 확인.")

random.seed(0)
print("무작위 미니배치 샘플:", buf.sample(2))

## 2. 목표가 계속 움직이는 문제, 타겟 네트워크로 고정하기 (책 7.2~7.3절)

`Q_net`과 `target_net`을 파이썬 딕셔너리로 흉내낸 아주 단순한
"신경망"으로 표현합니다 — 핵심은 신경망 구현이 아니라, **목표값 계산에
쓰이는 파라미터를 고정해두는 것**입니다.

In [ ]:
def make_q_table(n_states, n_actions, init=0.0):
    return {(s, a): init for s in range(n_states) for a in range(n_actions)}

def dqn_loss(Q_table, target_table, s, a, r, s_next, gamma, actions):
    target = r + gamma * max(target_table[(s_next, a2)] for a2 in actions)
    prediction = Q_table[(s, a)]
    return (target - prediction) ** 2

def should_update_target(step, update_freq):
    return step > 0 and step % update_freq == 0

print([should_update_target(s, 1000) for s in [999, 1000, 1500, 2000]])
assert [should_update_target(s, 1000) for s in [999, 1000, 1500, 2000]] == [False, True, False, True]

## 3. 타겟 네트워크가 실제로 '얼어있는' 것을 확인

In [ ]:
actions = [0, 1]
Q_table = make_q_table(2, 2, init=0.0)
target_table = dict(Q_table)  # 초기엔 동일하게 복사

update_freq = 3
transitions = [(0, 1, 1.0, 1), (1, 0, -1.0, 0), (0, 1, 1.0, 1), (1, 1, 5.0, 1)]

for step, (s, a, r, s_next) in enumerate(transitions, start=1):
    loss = dqn_loss(Q_table, target_table, s, a, r, s_next, gamma=0.9, actions=actions)
    Q_table[(s, a)] += 0.5 * (r + 0.9 * max(target_table[(s_next, a2)] for a2 in actions) - Q_table[(s, a)])
    print(f"step {step}: loss={loss:.3f}  Q_table={Q_table}  target_table={target_table}")
    if should_update_target(step, update_freq):
        target_table = dict(Q_table)  # theta -> theta^- 로 통째로 복사
        print(f"  -> step {step}: 타겟 네트워크를 현재 Q_table로 동기화!")